In [1]:
# ADHD-200 Parcellation using Glasser+Tian (gt_atlas_map)
# Output: (150 timepoints × 414 ROIs) fMRI + standardized phenotype
# ✅ Uses ALL available runs
# ✅ Robust to 3D/4D misclassification

import sys
import os

# Add gt_atlas directory to Python path
sys.path.append('/home/jaizor/jaizor/Ξ/gt_atlas')

import numpy as np
import pandas as pd
from pathlib import Path
from gt_atlas_map import GlasserTianParcellator
from nilearn import image
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION
# =============================================================================
ATLAS_DIR = '/home/jaizor/jaizor/Ξ/gt_atlas'
DATA_ROOT = Path('/home/jaizor/jaizor/xtra/data/nilearn_data/ADHD_athena')
PHENO_PATH = DATA_ROOT / "adhd200_subjects_with_fMRI.csv"
OUTPUT_DIR = DATA_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADHD_SITE_TR = {
    1: 2.0,   # Peking
    3: 2.0,   # KKI
    4: 2.0,   # NeuroIMAGE
    5: 2.0,   # NYU
    6: 2.5,   # OHSU
}

TARGET_TR = 2.0
TARGET_DURATION = 300.0  # → 150 timepoints

# =============================================================================
# HELPER: Safely get number of timepoints
# =============================================================================
def get_n_timepoints(img_path):
    """Return number of timepoints; 1 if 3D, T if 4D, 0 if invalid."""
    img = image.load_img(img_path)
    if len(img.shape) == 3:
        return 1
    elif len(img.shape) == 4:
        return img.shape[3]
    else:
        return 0

# =============================================================================
# MAIN PIPELINE
# =============================================================================
def main():
    logger.info("📥 Loading ADHD-200 phenotype...")
    df = pd.read_csv(PHENO_PATH)

    logger.info(f"✅ Starting with {len(df)} fMRI runs (all subjects × all runs).")

    # Rebuild fMRI paths
    def get_fMRI_path(row):
        sid = str(row["ScanDir ID"])
        run = int(row["run"])
        if run == 1:
            filename = f"{sid}.nii.gz"
        else:
            filename = f"{sid}_run{run}.nii.gz"
        full_path = DATA_ROOT / filename
        if full_path.exists():
            return str(full_path)
        else:
            alt_path = DATA_ROOT / f"{sid}_run{run}.nii.gz"
            return str(alt_path) if alt_path.exists() else None

    df["fMRI_path"] = df.apply(get_fMRI_path, axis=1)
    initial_n = len(df)
    df = df.dropna(subset=["fMRI_path"]).reset_index(drop=True)
    logger.info(f"✅ Kept {len(df)} runs with existing fMRI files (dropped {initial_n - len(df)}).")

    # 🔍 CRITICAL: Pre-screen for TRUE 4D fMRI with ≥20 timepoints
    logger.info("🔍 Pre-screening runs for valid 4D fMRI with ≥20 timepoints...")
    valid_mask = []
    fmri_paths_filtered = []
    tr_values_filtered = []

    for idx, row in df.iterrows():
        fp = row["fMRI_path"]
        tr = ADHD_SITE_TR.get(row["Site"], 2.0)
        try:
            n_tps = get_n_timepoints(fp)
            if n_tps >= 20:
                valid_mask.append(True)
                fmri_paths_filtered.append(fp)
                tr_values_filtered.append(tr)
            else:
                logger.warning(f"⚠️ Skipping {Path(fp).name}: not a valid fMRI time series (shape implies T={n_tps})")
                valid_mask.append(False)
        except Exception as e:
            logger.warning(f"⚠️ Skipping {Path(fp).name}: load error – {e}")
            valid_mask.append(False)

    df = df[valid_mask].reset_index(drop=True)
    fmri_paths = fmri_paths_filtered
    tr_values = np.array(tr_values_filtered, dtype=np.float32)
    logger.info(f"✅ Proceeding with {len(fmri_paths)} valid fMRI runs.")

    # Parcellate
    logger.info("🚀 Parcellating with Glasser+Tian atlases...")
    parcellator = GlasserTianParcellator(ATLAS_DIR)
    processed_data, valid_indices = parcellator.process_dataset(
        fmri_paths=fmri_paths,
        tr_values=tr_values,
        n_jobs=-1,
        target_tr=TARGET_TR,
        target_duration=TARGET_DURATION
    )

    valid_df = df.iloc[valid_indices].reset_index(drop=True)

    # Create run-level IDs
    def make_run_id(row):
        sid = str(row["ScanDir ID"])
        run = int(row["run"])
        return f"{sid}_run{run}" if run != 1 else sid

    valid_df["run_id"] = valid_df.apply(make_run_id, axis=1)

    # Save outputs
    data_array = np.stack(processed_data, axis=0).astype(np.float32)  # (N, 150, 414)
    run_ids = valid_df["run_id"].astype(str).values
    max_len = max(len(rid) for rid in run_ids)
    run_ids_safe = np.array(run_ids, dtype=f'U{max_len}')

    np.savez_compressed(
        OUTPUT_DIR / "fmri_ADHD_all_runs.npz",
        data=data_array,
        subject_ids=run_ids_safe
    )

    pheno_adhd = pd.DataFrame({
        'eid': valid_df["run_id"].astype(str),
        'subject_id': valid_df["ScanDir ID"].astype(str),
        'run': valid_df["run"].astype(int),
        'Age': valid_df["Age"],
        'Sex': valid_df["Gender"],
        'ADHD': (valid_df["DX"] != 0).astype(int)
    })
    pheno_adhd.to_csv(OUTPUT_DIR / "pheno_ADHD_all_runs.csv", index=False)

    # Final report
    logger.info("\n" + "=" * 60)
    logger.info("🎉 ADHD-200 Parcellation (All Runs) Complete!")
    logger.info(f"✅ Processed: {len(processed_data)} fMRI runs")
    logger.info(f"✅ Output shape per run: (150, 414)")
    logger.info("=" * 60)

    print("\n📊 Sample phenotype (run-level):")
    print(pheno_adhd.head(3))
    print(f"\nClass balance (ADHD=1): {pheno_adhd['ADHD'].value_counts().to_dict()}")
    print(f"\nTotal unique subjects: {pheno_adhd['subject_id'].nunique()}")


if __name__ == "__main__":
    main()

2025-12-14 20:13:52,200 - INFO - 📥 Loading ADHD-200 phenotype...
2025-12-14 20:13:52,209 - INFO - ✅ Starting with 1594 fMRI runs (all subjects × all runs).
2025-12-14 20:13:52,228 - INFO - ✅ Kept 1594 runs with existing fMRI files (dropped 0).
2025-12-14 20:13:52,229 - INFO - 🔍 Pre-screening runs for valid 4D fMRI with ≥20 timepoints...
2025-12-14 20:14:34,321 - WARNING - ⚠️ Skipping 1000804.nii.gz: not a valid fMRI time series (shape implies T=1)
2025-12-14 20:14:34,324 - WARNING - ⚠️ Skipping 1000804_run2.nii.gz: not a valid fMRI time series (shape implies T=1)
2025-12-14 20:14:34,327 - WARNING - ⚠️ Skipping 1000804.nii.gz: not a valid fMRI time series (shape implies T=1)
2025-12-14 20:14:34,330 - WARNING - ⚠️ Skipping 1000804_run2.nii.gz: not a valid fMRI time series (shape implies T=1)
2025-12-14 20:14:34,332 - WARNING - ⚠️ Skipping 1000804.nii.gz: not a valid fMRI time series (shape implies T=1)
2025-12-14 20:14:34,334 - WARNING - ⚠️ Skipping 1000804_run2.nii.gz: not a valid fMRI 


📊 Sample phenotype (run-level):
       eid subject_id  run    Age  Sex  ADHD
0  1017176    1017176    1  11.66  0.0     0
1  1125505    1125505    1  19.30  1.0     0
2  1208586    1208586    1  20.89  1.0     1

Class balance (ADHD=1): {0: 139, 1: 103}

Total unique subjects: 242
